In [3]:
# Cell 1: Imports and Configurations
import os
import pickle
import numpy as np
import torch
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.svm import SVC
from config import PROCESSED_DIR, MODEL_PATH, CLASSES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(keep_all=False, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

C:\Users\yalim\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Cell 2: Feature Extraction Loop
X = []
y = []

for class_idx, class_name in enumerate(CLASSES):
    class_dir = os.path.join(PROCESSED_DIR, class_name)
    if not os.path.exists(class_dir):
        continue
    for filename in os.listdir(class_dir):
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        img_path = os.path.join(class_dir, filename)
        try:
            img = Image.open(img_path).convert('RGB')
            with torch.no_grad():
                face = mtcnn(img)
                if face is not None:
                    embedding = resnet(face.unsqueeze(0).to(device)).cpu().numpy()[0]
                    X.append(embedding)
                    y.append(class_idx)
        except Exception as e:
            print(f"Skipping image {filename}: {e}")

X = np.array(X)
y = np.array(y)
print(f"Extracted {len(X)} face embeddings.")

Extracted 100 face embeddings.


In [9]:
# Cell 3: Train Linear Probe Classifier
if len(np.unique(y)) < 2:
    print("Error: You need at least two distinct classes containing images to train.")
else:
    classifier = SVC(C=1.0, kernel='linear', probability=True)
    classifier.fit(X, y)
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(classifier, f)
    print(f"Model successfully saved to {MODEL_PATH}")

Model successfully saved to d:\code projects\cute-couple-finder\linear_probe_model.pkl
